**Import Required Libraries**

In [ ]:
import pandas as pd
import numpy as np

 **Load the Dataset**

In [ ]:
df=pd.read_csv("C:\Users\91982\OneDrive\Desktop\End-To-End\Email-Spam-detection-using-NLP\spam_ham_dataset.csv")
df.head(5)

,Unnamed: 0,label,text,label_num
0,605,ham,Subject: enron methanol ; meter # : 988291\r\n...,0
1,2349,ham,"Subject: hpl nom for january 9 , 2001\r\n( see...",0
2,3624,ham,"Subject: neon retreat\r\nho ho ho , we ' re ar...",0
3,4685,spam,"Subject: photoshop , windows , office . cheap ...",1
4,2030,ham,Subject: re : indian springs\r\nthis deal is t...,0


**Data Cleaning**

In [ ]:
df=df.drop("Unnamed: 0",axis=1)

In [ ]:
df=df.drop("label_num",axis=1)

**Download NLTK Resources**

In [ ]:
import nltk
from nltk.stem import PorterStemmer
from nltk.corpus import stopwords
import re

ps = PorterStemmer()

**Text Preprocessing**

In [ ]:
nltk.download('stopwords')
corpus=[]
for i in range(0,len(df)):
  review=re.sub('[^a-zA-Z]',' ',df['text'][i])
  review=review.lower()
  review=review.split()
  review=[ps.stem(word) for word in review if not word in stopwords.words('english')]
  review=' '.join(review)
  corpus.append(review)

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


In [ ]:
corpus

['subject enron methanol meter follow note gave monday preliminari flow data provid daren pleas overrid pop daili volum present zero reflect daili activ obtain ga control chang need asap econom purpos',
 'subject hpl nom januari see attach file hplnol xl hplnol xl',
 'subject neon retreat ho ho ho around wonder time year neon leader retreat time know time year extrem hectic tough think anyth past holiday life go past week decemb januari like think minut calend hand begin fall semest retreat schedul weekend januari youth minist confer brad dustin connect week go chang date follow weekend januari come part need think think agre import us get togeth time recharg batteri get far spring semest lot troubl difficult us get away without kid etc brad came potenti altern get togeth weekend let know prefer first option would retreat similar done past sever year year could go heartland countri inn www com outsid brenham nice place bedroom bedroom hous side side countri real relax also close brenha

In [ ]:
y=pd.get_dummies(df['label'])
y=y.iloc[:,1].values

**Feature Extraction using Bag of Words**

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer
cv=CountVectorizer(max_features=2500)
x=cv.fit_transform(corpus).toarray()

**Split the Dataset**

In [ ]:
from sklearn.model_selection import train_test_split
x_train,x_test,y_train,y_test=train_test_split(x,y,test_size=0.2,random_state=42)

In [ ]:
print(x_train[0])
print(y_train[0])

[0 0 0 ... 0 0 0]
False


**Train Logistic Regression Model and evaluation of model**

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import confusion_matrix,accuracy_score,classification_report
count_model=LogisticRegression()
count_model.fit(x_train,y_train)
y_pred=count_model.predict(x_test)
print(accuracy_score(y_test,y_pred))
print(confusion_matrix(y_test,y_pred))
print(classification_report(y_test,y_pred))

0.9632850241545894
[[717  25]
 [ 13 280]]
              precision    recall  f1-score   support

       False       0.98      0.97      0.97       742
        True       0.92      0.96      0.94       293

    accuracy                           0.96      1035
   macro avg       0.95      0.96      0.96      1035
weighted avg       0.96      0.96      0.96      1035



**Feature Extraction using TF-IDF**

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
tcv=TfidfVectorizer(max_features=2500)
x_tfidf=tcv.fit_transform(corpus).toarray()

In [ ]:
xt_train,xt_test,yt_train,yt_test=train_test_split(x_tfidf,y,test_size=0.2,random_state=42)

**Train and Evaluate TF-IDF Model**

In [ ]:
tfidf_model=LogisticRegression()
tfidf_model.fit(xt_train,yt_train)
y_pred=tfidf_model.predict(xt_test)
print(accuracy_score(yt_test,y_pred))
print(confusion_matrix(yt_test,y_pred))
print(classification_report(yt_test,y_pred))

0.9768115942028985
[[729  13]
 [ 11 282]]
              precision    recall  f1-score   support

       False       0.99      0.98      0.98       742
        True       0.96      0.96      0.96       293

    accuracy                           0.98      1035
   macro avg       0.97      0.97      0.97      1035
weighted avg       0.98      0.98      0.98      1035



**Install and Load Word2Vec Model**

In [ ]:
!pip install gensim
import gensim
from gensim.models import word2vec,keyedvectors
import gensim.downloader as api

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.9/27.9 MB 65.8 MB/s eta 0:00:00


**Word2Vec Embeddings**

In [ ]:
wv=api.load('word2vec-google-news-300')

[==================================================] 100.0% 1662.8/1662.8MB downloaded


In [ ]:
from nltk.stem import WordNetLemmatizer
lemmatizer=WordNetLemmatizer()

In [ ]:
import nltk
nltk.download('wordnet')
corpus=[]
for i in range(0,len(df)):
  review=re.sub('[^a-zA-Z]',' ',df['text'][i])
  review=review.lower()
  review=review.split()
  review=[lemmatizer.lemmatize(word) for word in review if not word in stopwords.words('english')]
  review=' '.join(review)
  corpus.append(review)

[nltk_data] Downloading package wordnet to /root/nltk_data...


In [ ]:
from nltk import sent_tokenize
from gensim.utils import simple_preprocess

In [ ]:
import nltk
nltk.download('punkt_tab')
words=[]
for sent in corpus:
  sent_token=sent_tokenize(sent)
  for sent in sent_token:
    words.append(simple_preprocess(sent))

[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


In [ ]:
import gensim

w2v_model = gensim.models.Word2Vec(
    words,
    window=5,
    min_count=2
)

**Create Average Word2Vec Vectors**

In [ ]:
def avg_word2vec(doc):
    vectors = [w2v_model.wv[word] for word in doc if word in w2v_model.wv]

    if len(vectors) == 0:
        return np.zeros(w2v_model.vector_size)

    return np.mean(vectors, axis=0)

In [ ]:
!pip install tqdm

In [ ]:
from tqdm import tqdm

In [ ]:
x = []

for sentence in words:
    x.append(avg_word2vec(sentence))

x = np.array(x)

In [ ]:
x_new=np.array(x)

**Train Logistic Regression on Word2Vec Features**

In [ ]:
from sklearn.model_selection import train_test_split
x_train,x_test,y_train,y_test=train_test_split(x_new,y,test_size=0.2,random_state=42)

In [ ]:
from sklearn.linear_model import LogisticRegression

x_train, x_test, y_train, y_test = train_test_split(
    x, y, test_size=0.2, random_state=42
)

avg_lr_model = LogisticRegression()

avg_lr_model.fit(x_train, y_train)

y_pred = avg_lr_model.predict(x_test)

In [ ]:
import pickle
import os

# Create the 'models' directory if it doesn't exist
os.makedirs('models', exist_ok=True)

pickle.dump(count_model, open("models/count_model.pkl", "wb"))
pickle.dump(cv, open("models/count_vectorizer.pkl", "wb"))
pickle.dump(tfidf_model, open("models/tfidf_model.pkl", "wb"))
pickle.dump(tcv, open("models/tfidf_vectorizer.pkl", "wb"))
pickle.dump(avg_w2v_model, open("models/avg_word2vec_model.pkl", "wb"))


In [ ]:
import os

print(os.getcwd())

/content


In [ ]:
import pickle

# Save Logistic Regression classifier
pickle.dump(avg_lr_model,
            open("models/avg_word2vec_lr.pkl", "wb"))

# Save Word2Vec embeddings
w2v_model.save("models/word2vec.model")

In [ ]:
from google.colab import files

files.download("/content/models/count_model.pkl")
files.download("/content/models/count_vectorizer.pkl")

files.download("/content/models/tfidf_model.pkl")
files.download("/content/models/tfidf_vectorizer.pkl")

files.download("/content/models/avg_word2vec_lr.pkl")      # or avg_word2vec_model.pkl (whichever you saved)

files.download("/content/models/word2vec.model")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

...

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>